### Dataset and Task Metadata

In [1]:
from data_foundry.schema import DatasetMetadata, PredictiveMLTaskMetadata

dataset_mold = DatasetMetadata(
    unique_name="porto_seguro",
    dataset_year="2017",
    domain_str="insurance",
    # Data Source
    dataset_source="Kaggle",
    original_dataset_source_download_link="https://www.kaggle.com/c/porto-seguro-safe-driver-prediction",
    download_description="""
We use the train.csv from the Kaggle competition.

kaggle competitions download -c porto-seguro-safe-driver-prediction -f train.csv && unzip train.csv.zip &&  rm train.csv.zip
mkdir -p local-data-warehouse/porto_seguro && mv train.csv local-data-warehouse/porto_seguro/
""",
    # References
    academic_reference_bibtex=r"""@misc{Howard2017PortoSegurosSafeDriverPrediction,
  author = {Addison Howard and Adriano Moala and Walter Reade},
  title  = {Porto Seguro’s Safe Driver Prediction},
  year   = {2017},
  howpublished = {\url{https://kaggle.com/competitions/porto-seguro-safe-driver-prediction}},
  note   = {Kaggle competition}
}
""",
    academic_reference_bibtex_key="Howard2017PortoSegurosSafeDriverPrediction",
    license="Kaggle Competition Rules",
    data_tags=["IID", "Anonymized"],
    curation_comments="""
We start with the train.csv from Kaggle.

- The data has been anonymized, so feature meanings are unknown.
- Following the expert solutions on Kaggle, we drop *calc features.
- In general, the 2nd and 3rd place did a lot of interesting feature engineering for interactions. We do not copy these as the data is clean as it is.
- We drop the index as it seems to be uninformative for this dataset.
- There exist only 6 duplicates in the training data. We drop them as the portion is too small to be meaningful.
""",
)
task_mold = PredictiveMLTaskMetadata(
    target_column_name="target",
    problem_type="binary_classification",
    objective_metric_name="normalized_gini_coefficient", # see https://www.kaggle.com/c/ClaimPredictionChallenge/discussion/703
    stratify_on="target",
)

## Preprocessing

In [2]:
import pandas as pd
import numpy as np

df = pd.read_csv(dataset_mold.path / "train.csv")
print("Loaded data shape:", df.shape)

# Create real nan values
df = df.replace(-1, np.nan)

# Drop *calc features
calc_features = [col for col in df.columns if "calc" in col]
df = df.drop(columns=calc_features)

# Drop index
df = df.drop(columns=["id"])

# Dtypes
as_cat_type = [
    col for col in df.columns if col.endswith("cat") or col.endswith("bin")
]
df[as_cat_type] = df[as_cat_type].astype("category")
df[task_mold.target_column_name] = df[task_mold.target_column_name].astype("category")

# Drop duplicates w/o target
df = df.drop_duplicates(subset=[col for col in df.columns if col != task_mold.target_column_name])
df = df.reset_index(drop=True)

Loaded data shape: (595212, 59)


## Data Checks

In [3]:
from data_foundry import dataset_checks
df_head, summary, numeric_stats, cat_stats, target_df = dataset_checks.run_all_checks(
    data=df,
    classification=task_mold.is_classification,
    target_feature=task_mold.target_column_name,
    print_report=False, # In notebook...
)


#### Dataset Overview
Rows: 595,206
Columns: 38
Use sampling: False (sample size: 595,206)


Get row duplicates (staged, merged)...
Using top-10 columns for initial filtering: ['ps_car_13', 'ps_reg_03', 'ps_car_14', 'ps_car_12', 'ps_car_11_cat', 'ps_reg_02', 'ps_car_06_cat', 'ps_car_15', 'ps_ind_15', 'ps_car_01_cat']
Rows remaining as candidates after top-10 filter: 3,702 (of 595,206)

#### Duplicate Report
Total duplicate rows: 0 (0.00% of dataset)
Duplicate rows ignoring target: 0 (0.00% of dataset)
Get column duplicates...
Duplicate columns: 0 (0.00% of columns)

Data quality checks completed.


In [4]:
# Sample Rows
df_head

,target,ps_ind_01,ps_ind_02_cat,ps_ind_03,ps_ind_04_cat,ps_ind_05_cat,ps_ind_06_bin,ps_ind_07_bin,ps_ind_08_bin,ps_ind_09_bin,ps_ind_10_bin,ps_ind_11_bin,ps_ind_12_bin,ps_ind_13_bin,ps_ind_14,ps_ind_15,ps_ind_16_bin,ps_ind_17_bin,ps_ind_18_bin,ps_reg_01,ps_reg_02,ps_reg_03,ps_car_01_cat,ps_car_02_cat,ps_car_03_cat,ps_car_04_cat,ps_car_05_cat,ps_car_06_cat,ps_car_07_cat,ps_car_08_cat,ps_car_09_cat,ps_car_10_cat,ps_car_11_cat,ps_car_11,ps_car_12,ps_car_13,ps_car_14,ps_car_15
0,0,2,2.0,5,1.0,0.0,0,1,0,0,0,0,0,0,0,11,0,1,0,0.7,0.2,0.718070,10.0,1.0,NaN,0,1.0,4,1.0,0,0.0,1,12,2.0,0.400000,0.883679,0.370810,3.605551
1,0,1,1.0,7,0.0,0.0,0,0,1,0,0,0,0,0,0,3,0,0,1,0.8,0.4,0.766078,11.0,1.0,NaN,0,NaN,11,1.0,1,2.0,1,19,3.0,0.316228,0.618817,0.388716,2.449490
2,0,5,4.0,9,1.0,0.0,0,0,1,0,0,0,0,0,0,12,1,0,0,0.0,0.0,NaN,7.0,1.0,NaN,0,NaN,14,1.0,1,2.0,1,60,1.0,0.316228,0.641586,0.347275,3.316625
3,0,0,1.0,2,0.0,0.0,1,0,0,0,0,0,0,0,0,8,1,0,0,0.9,0.2,0.580948,7.0,1.0,0.0,0,1.0,11,1.0,1,3.0,1,104,1.0,0.374166,0.542949,0.294958,2.000000
4,0,0,2.0,0,1.0,0.0,1,0,0,0,0,0,0,0,0,9,1,0,0,0.7,0.6,0.840759,11.0,1.0,NaN,0,NaN,14,1.0,1,2.0,1,82,3.0,0.316070,0.565832,0.365103,2.000000


In [5]:
# Feature Summary
summary

,index,dtype,n_missing,pct_missing,n_unique,examples
0,ps_car_03_cat,category,411225.0,69.09,2.0,"1.0, 0.0"
1,ps_car_05_cat,category,266546.0,44.78,2.0,"1.0, 0.0"
2,ps_car_07_cat,category,11489.0,1.93,2.0,"1.0, 0.0"
3,ps_ind_05_cat,category,5809.0,0.98,7.0,"0.0, 6.0, 4.0, 1.0, 3.0, 2.0, 5.0"
4,ps_car_09_cat,category,569.0,0.10,5.0,"2.0, 0.0, 1.0, 3.0, 4.0"
5,ps_ind_02_cat,category,216.0,0.04,4.0,"1.0, 2.0, 3.0, 4.0"
6,ps_car_01_cat,category,107.0,0.02,12.0,"11.0, 7.0, 6.0, 10.0, 4.0, 9.0, 5.0, 8.0, 3.0, 0.0"
7,ps_ind_04_cat,category,83.0,0.01,2.0,"0.0, 1.0"
8,target,category,0.0,0.00,2.0,"0, 1"
9,ps_ind_06_bin,category,0.0,0.00,2.0,"0, 1"


In [6]:
# Numeric Feature Statistics
numeric_stats

,count,mean,std,min,max
ps_ind_01,595206.0,1.900396,1.983791,0.000000,7.000000
ps_ind_03,595206.0,4.423332,2.699906,0.000000,11.000000
ps_ind_14,595206.0,0.012451,0.127546,0.000000,4.000000
ps_ind_15,595206.0,7.299942,3.546035,0.000000,13.000000
ps_reg_01,595206.0,0.610995,0.287640,0.000000,0.900000
ps_reg_02,595206.0,0.439187,0.404265,0.000000,1.800000
ps_reg_03,487439.0,0.894047,0.345413,0.061237,4.037945
ps_car_11,595201.0,2.346095,0.832497,0.000000,3.000000
ps_car_12,595205.0,0.379947,0.058300,0.100000,1.264911
ps_car_13,595206.0,0.813265,0.224589,0.250619,3.720626


In [7]:
# Categorical Feature Statistics
cat_stats

value   count    pct
column        rank                     
ps_car_01_cat 1     11.0  207573  34.87
              2      7.0  179242  30.11
              3      6.0   62393  10.48
              4     10.0   50086   8.41
              5      4.0   26174   4.40
ps_car_02_cat 1      1.0  493984  82.99
              2      0.0  101217  17.01
              3     <NA>       5   0.00
ps_car_03_cat 1     <NA>  411225  69.09
              2      1.0  110709  18.60
              3      0.0   73272  12.31
ps_car_04_cat 1        0  496575  83.43
              2        1   32115   5.40
              3        2   23770   3.99
              4        8   20598   3.46
              5        9   19034   3.20
ps_car_05_cat 1     <NA>  266546  44.78
              2      1.0  172667  29.01
              3      0.0  155993  26.21
ps_car_06_cat 1       11  131526  22.10
              2        1  118384  19.89
              3        0  110417  18.55
              4       14   59253   9.96
              5       10   33466   5.62
ps_car_07_cat 1      1.0  553142  92.93
              2      0.0   30575   5.14
              3     <NA>   11489   1.93
ps_car_08_cat 1        1  495259  83.21
              2        0   99947  16.79
ps_car_09_cat 1      2.0  353477  59.39
              2      0.0  194517  32.68
              3      1.0   29080   4.89
              4      3.0   14756   2.48
              5      4.0    2807   0.47
ps_car_10_cat 1        1  590173  99.15
              2        0    4857   0.82
              3        2     176   0.03
ps_car_11_cat 1      104   85083  14.29
              2      103   24262   4.08
              3       64   22278   3.74
              4       87   17106   2.87
              5       32   12576   2.11
ps_ind_02_cat 1      1.0  431854  72.56
              2      2.0  123572  20.76
              3      3.0   28186   4.74
              4      4.0   11378   1.91
              5     <NA>     216   0.04
ps_ind_04_cat 1      0.0  346960  58.29
              2      1.0  248163  41.69
              3     <NA>      83   0.01
ps_ind_05_cat 1      0.0  528003  88.71
              2      6.0   20662   3.47
              3      4.0   18344   3.08
              4      1.0    8322   1.40
              5      3.0    8233   1.38
ps_ind_06_bin 1        0  360849  60.63
              2        1  234357  39.37
ps_ind_07_bin 1        0  442219  74.30
              2        1  152987  25.70
ps_ind_08_bin 1        0  497638  83.61
              2        1   97568  16.39
ps_ind_09_bin 1        0  484912  81.47
              2        1  110294  18.53
ps_ind_10_bin 1        0  594984  99.96
              2        1     222   0.04
ps_ind_11_bin 1        0  594199  99.83
              2        1    1007   0.17
ps_ind_12_bin 1        0  589588  99.06
              2        1    5618   0.94
ps_ind_13_bin 1        0  594642  99.91
              2        1     564   0.09
ps_ind_16_bin 1        1  393326  66.08
              2        0  201880  33.92
ps_ind_17_bin 1        0  523137  87.89
              2        1   72069  12.11
ps_ind_18_bin 1        0  503875  84.66
              2        1   91331  15.34
target        1        0  573513  96.36
              2        1   21693   3.64

In [8]:
# Target Distribution
target_df

,count,pct
target,,
0,573513,96.36
1,21693,3.64


## Task Curation

In [9]:
from data_foundry.curation_recommendations import get_recommended_splits_dimensions

n_repeats, n_splits, none_or_test_size = get_recommended_splits_dimensions(dataset=df)
print(f"Recommended IID splits: n_repeats={n_repeats}, n_splits={n_splits}, test_size={none_or_test_size}")

Recommended IID splits: n_repeats=1, n_splits=3, test_size=None


In [10]:
from data_foundry.schema import PredictiveMLSplitsMetadata
from data_foundry.curation_recommendations import get_recommended_iid_splits

splits_mold = PredictiveMLSplitsMetadata(
    splits_comment="Default splits for IID data.",
    splits=get_recommended_iid_splits(
        dataset=df,
        n_repeats=n_repeats,
        n_splits=n_splits,
        test_size=none_or_test_size,
        stratify_on=task_mold.stratify_on,
    ),
)

Using Stratified IID splits.


## Export

In [11]:
from data_foundry.curation_container import CuratedContainer
curated_data = CuratedContainer(
    dataset=df,
    dataset_metadata=dataset_mold,
    task_metadata=task_mold,
    experiment_metadata=splits_mold,
 )
curated_data.save()
print(curated_data.uuid)
print(curated_data.checksum)

Calculating checksum for curated container...


Saving curated container to porto_seguro/019d5dc5-d2df-75ec-a9d1-b1fd8dec495d


019d5dc5-d2df-75ec-a9d1-b1fd8dec495d
48c52a17f0e1afa8458f9df78b6ee521f144c1ed0688e8cfe764844488c8830d
